# Prueba de ingeniería de datos: pipeline por micro batches

**Objetivo:** cargar los archivos CSV en PostgreSQL por partes (micro batches) y llevar unas estadísticas
del campo `price` que se actualizan con cada fila cargada, sin volver a consultar los datos ya cargados.

**Herramientas:** Python, pandas (lectura por bloques), psycopg2 (conexión a PostgreSQL) y PostgreSQL en Docker.

**Cómo ejecutarlo:**
1. Levantar la base de datos: `docker compose up -d`
2. Instalar las librerías: `pip install -r requirements.txt`
3. Ejecutar todas las celdas en orden (*Run All*).

## 1. Conexión a la base de datos

Me conecto a PostgreSQL, que corre en Docker y expone el puerto 5434.
El cursor (`cur`) es el objeto que envía las sentencias SQL a la base.

In [1]:
import pandas as pd
import psycopg2

conn = psycopg2.connect(host="localhost", port=5434, dbname="reto", user="pragma", password="pragma")
cur = conn.cursor()

OperationalError: connection to server at "localhost" (::1), port 5434 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5434 failed: Connection refused (0x0000274D/10061)
	Is the server running on that host and accepting TCP/IP connections?


## 2. Creación de la tabla

Una sola tabla guarda las transacciones de todos los archivos.

- `price` puede quedar vacío (`NULL`), porque algunas filas de los CSV vienen sin precio.
- `archivo` guarda de qué CSV vino cada fila.

La tabla se borra y se crea de nuevo al inicio, para que el notebook siempre empiece desde cero.

In [ ]:
cur.execute("DROP TABLE IF EXISTS transacciones")
cur.execute("""
    CREATE TABLE transacciones (
        fecha    DATE,
        price    NUMERIC,
        user_id  INTEGER,
        archivo  TEXT
    )
""")
conn.commit()

## 3. Estadísticas en ejecución

En vez de calcular `SELECT AVG(price)` después de cada carga, llevo una "libreta" en memoria con cinco valores
y la actualizo con cada fila nueva:

| Valor | Cómo se actualiza |
|---|---|
| `filas` | +1 con cada fila |
| `con_precio` | +1 solo si la fila trae precio |
| `suma` | se le suma el precio |
| `min` / `max` | se compara con el precio nuevo |

El promedio no se guarda: se calcula como `suma / con_precio`.
Las filas sin precio cuentan como fila cargada, pero no entran en el promedio (igual que `AVG` en SQL).

In [ ]:
estadisticas = {"filas": 0, "con_precio": 0, "suma": 0.0, "min": None, "max": None}


def actualizar_estadisticas(precio):
    estadisticas["filas"] += 1
    if precio is None:
        return
    estadisticas["con_precio"] += 1
    estadisticas["suma"] += precio
    if estadisticas["min"] is None or precio < estadisticas["min"]:
        estadisticas["min"] = precio
    if estadisticas["max"] is None or precio > estadisticas["max"]:
        estadisticas["max"] = precio


def mostrar_estadisticas(titulo):
    promedio = estadisticas["suma"] / estadisticas["con_precio"]
    print(f"{titulo}: filas={estadisticas['filas']}, promedio={promedio:.4f}, min={estadisticas['min']}, max={estadisticas['max']}")

## 4. Carga de un archivo por micro batches

Se lee el archivo de 10 en 10 filas, así que nunca hay un archivo completo en memoria.
Para cada fila:

1. Convierto la fecha (`1/10/2012` está en formato mes/día/año).
2. Si el precio viene vacío, pandas lo marca como `NaN`; lo convierto a `None` para que se guarde como `NULL`.
   Si se guardara `NaN`, el promedio de la base de datos daría `NaN`.
3. Inserto la fila en la base de datos.
4. Actualizo las estadísticas en ejecución.

Al terminar el archivo, `conn.commit()` confirma los datos en la base.

In [ ]:
def cargar_archivo(archivo):
    ruta = "dataPruebaDataEngineer/" + archivo
    for bloque in pd.read_csv(ruta, chunksize=10):
        for _, fila in bloque.iterrows():
            fecha = pd.to_datetime(fila["timestamp"], format="%m/%d/%Y").date()
            precio = None if pd.isna(fila["price"]) else float(fila["price"])
            user_id = int(fila["user_id"])

            cur.execute(
                "INSERT INTO transacciones (fecha, price, user_id, archivo) VALUES (%s, %s, %s, %s)",
                (fecha, precio, user_id, archivo),
            )
            actualizar_estadisticas(precio)
    conn.commit()

## 5. Carga de los 5 archivos

Cargo los archivos en orden cronológico (excepto `validation.csv`) y muestro las estadísticas después de cada uno.

In [ ]:
archivos = ["2012-1.csv", "2012-2.csv", "2012-3.csv", "2012-4.csv", "2012-5.csv"]

for archivo in archivos:
    cargar_archivo(archivo)
    mostrar_estadisticas(archivo)

2012-1.csv: filas=22, promedio=59.6500, min=14.0, max=97.0
2012-2.csv: filas=51, promedio=56.7959, min=10.0, max=100.0
2012-3.csv: filas=82, promedio=57.9125, min=10.0, max=100.0
2012-4.csv: filas=112, promedio=57.7778, min=10.0, max=100.0
2012-5.csv: filas=143, promedio=57.8849, min=10.0, max=100.0


## 6. Comprobación en la base de datos

Consulto la base directamente. Aquí sí uso `COUNT`, `AVG`, `MIN` y `MAX`, porque es solo para comprobar
que las estadísticas en ejecución son correctas: los valores deben coincidir con los del paso 5.

In [ ]:
def consultar_base():
    cur.execute("SELECT COUNT(*), AVG(price), MIN(price), MAX(price) FROM transacciones")
    filas, promedio, minimo, maximo = cur.fetchone()
    print(f"Base de datos: filas={filas}, promedio={promedio:.4f}, min={minimo}, max={maximo}")


consultar_base()

Base de datos: filas=143, promedio=57.8849, min=10.0, max=100.0


## 7. Validación

Paso `validation.csv` por el mismo pipeline y vuelvo a consultar la base para ver cómo cambiaron los valores.

In [ ]:
cargar_archivo("validation.csv")
mostrar_estadisticas("validation.csv")
consultar_base()

validation.csv: filas=151, promedio=57.0068, min=10.0, max=100.0
Base de datos: filas=151, promedio=57.0068, min=10.0, max=100.0


## 8. Conclusiones

- Las estadísticas en ejecución coinciden con la consulta a la base de datos, antes y después de la validación.
- El promedio bajó por que validation sumo 8 valores mas y afecto el promedio
- El minimo y el maximo no cambiaron por que dentro de validation no existen valores menores ni mayores
- El problema para miles de filas lo unico que cambiaria seria la cantidad de filas en los lotes que se envian, y si son millones, se tendria que usar spark para realizar procesamiento distribuido

In [ ]:
cur.close()
conn.close()